# Sentence Embeddings

## Definition
Sentence embeddings are dense, fixed-length vector representations of entire sentences that encode their semantic meaning. Unlike word embeddings, they capture the combined meaning of all words in a sentence in a single vector. Modern sentence embedding models such as Sentence-BERT (SBERT) use transformer architectures fine-tuned specifically for sentence-level semantic similarity tasks.

## Why It Is Needed
- **Sentence-Level Semantics:** Captures the meaning of a full sentence, not just individual words — critical for tasks like semantic search and paraphrase detection.
- **Efficient Similarity Search:** Encodes sentences into comparable vectors, enabling fast large-scale semantic search using cosine or dot-product similarity.
- **Limitations of Word Averaging:** Simple averaging of word embeddings loses word order and sentence-level context — dedicated sentence models address this.

## Real-World Applications
- Semantic search engines that retrieve meaning, not just keywords
- Paraphrase detection and duplicate question identification (e.g., StackOverflow, Quora)
- Sentence clustering for topic discovery in large text collections
- Natural Language Inference (NLI) — entailment, contradiction, neutral classification
- FAQ matching and question answering retrieval systems

## Important Points
- **Sentence-BERT (SBERT):** Fine-tunes BERT using siamese and triplet network structures on NLI and STS datasets for semantic similarity.
- **`sentence-transformers` Library:** The standard Python library for sentence embeddings — supports 100+ pre-trained models.
- **Mean Pooling:** The most common method to derive a sentence vector from transformer token outputs — averages all token embeddings.
- **[CLS] Token:** In BERT, the [CLS] token representation is often used as a sentence embedding, though mean pooling generally performs better.
- **STS (Semantic Textual Similarity) Benchmark:** Standard evaluation dataset for sentence embedding quality.
- **Cosine Similarity** is the standard metric for comparing sentence embedding pairs.

## Visual Understanding
```
[Insert diagram of Sentence-BERT's siamese network architecture:
 Sentence A → BERT → Mean Pooling → Vector A
 Sentence B → BERT → Mean Pooling → Vector B
 Cosine Similarity(Vector A, Vector B) → Similarity Score]
```

## Implementation
Practical implementation will be added here.

## Key Takeaways
- Sentence embeddings encode the full semantic meaning of a sentence in a single dense vector.
- SBERT is the state-of-the-art model for sentence-level semantic similarity.
- Mean pooling over BERT token outputs outperforms using the [CLS] token for sentence embeddings.
- The `sentence-transformers` library provides easy access to 100+ pre-trained sentence models.
- Sentence embeddings power modern semantic search, FAQ matching, and paraphrase detection systems.

In [3]:
import spacy 
import numpy as np 

nlp = spacy.load("en_core_web_sm")


def average_embedding(sentence):
    doc = nlp(sentence)

    vectors = ([token.vector for token in doc if token.has_vector])
    #print(vectors)

    return np.mean(vectors , axis = 0) if vectors else np.zeros((nlp.vocab.vectors_length,))

sentence = "the dog runs fast"

embedding = average_embedding(sentence)

print(embedding[:5])

[ 0.22581619 -0.11688705 -0.4035039  -0.08471712 -0.05002381]


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# sample corpus

corpus = [
    "the dog runs fast",
    "A cat sleeps on the sofa",
    "Dogs and Cats are great pets"
]
# Compute TF-IDF scores
vectorizer = TfidfVectorizer()
vectorizer.fit(corpus)
tfidf_scores = vectorizer.transform([sentence]).toarray()[0]
words = vectorizer.get_feature_names_out()
print(words)
# Compute weighted embeddings
def tfidf_weighted_embedding(sentence):
    doc = nlp(sentence)
    word_embeddings = []
    weights = []

    for token in doc:
        word = token.text.lower()
        if word in words and token.has_vector:
            idx = list(words).index(word)
            word_embeddings.append(token.vector * tfidf_scores[idx])
            weights.append(tfidf_scores[idx])

    return np.sum(word_embeddings, axis=0) / np.sum(weights) if weights else np.zeros((nlp.vocab.vectors_length,))

embedding = tfidf_weighted_embedding(sentence)

print("Sentence Embedding (TF-IDF Weighted):", embedding[:5])  # Print first 5 values



['and' 'are' 'cat' 'cats' 'dog' 'dogs' 'fast' 'great' 'on' 'pets' 'runs'
 'sleeps' 'sofa' 'the']
Sentence Embedding (TF-IDF Weighted): [ 0.19531655 -0.09783761 -0.43420076 -0.13278286 -0.02624961]


In [7]:
from sklearn.decomposition import PCA

# Generate embeddings for all sentences in corpus
sentence_embeddings = np.array([average_embedding(sent) for sent in corpus])

# Apply PCA to reduce dimensions (e.g., from 300 to 50)
pca = PCA()
reduced_embeddings = pca.fit_transform(sentence_embeddings)

print("Reduced Sentence Embedding (PCA):", reduced_embeddings[0][:5])  # Print first 5 values

Reduced Sentence Embedding (PCA): [ 1.1098539e+00 -1.4632885e+00 -1.4943177e-07]


In [9]:
!pip install sentence-transformers

   ---------------------------------------- 0.0/596.4 kB ? eta -:--:--
   ---------------------------------------- 596.4/596.4 kB 4.3 MB/s  0:00:00


In [10]:
from sentence_transformers import SentenceTransformer

# Load pre-trained SBERT model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Compute SBERT sentence embedding
sentence = "The dog runs fast"
embedding = model.encode(sentence)

print("Sentence Embedding (SBERT):", embedding[:5])  # Print first 5 values

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\Natural_Language_procesing\venv\lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence Embedding (SBERT): [0.0323461  0.03101398 0.01024306 0.0383051  0.0015568 ]


# NLP task - sentence similarity

In [11]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load pre-trained SBERT model
model = SentenceTransformer("all-MiniLM-L6-v2")

def compute_similarity(sentence1, sentence2):
    # Encode sentences into embeddings
    embedding1 = model.encode(sentence1)
    embedding2 = model.encode(sentence2)

    # Compute cosine similarity
    similarity_score = cosine_similarity([embedding1], [embedding2])[0][0]

    return similarity_score

# Example sentences
sentence1 = "A dog is running in the park ."
sentence2 = "A pup is playing in the park."

# Compute similarity
similarity = compute_similarity(sentence1, sentence2)

print(f"Sentence Similarity Score (SBERT): {similarity:.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sentence Similarity Score (SBERT): 0.8210
